# clean-index.ipynb

This file runs locally. It builds a clean table from the typed LABINS volume index pages in `data/labins/index-ocr/`, one row per township and range, with columns for surveyor, variation, diagram page, TWP and RGE. Only pages tagged `index` are used, not `unsure` or `diagram`.

The first step is the surveyor column. Each row can have more than one surveyor, so the clean value is a set. The first cell extracts the raw surveyor lines for each row. The second cell lists every surveyor name variant by frequency, then clusters them, to code the correct names and match the rest by similarity.

In [8]:
# imports
import re, hashlib, statistics, json
from pathlib import Path
import numpy as np
import pandas as pd

# set dir
OCR_DIR = Path("data/labins/index-ocr")
OUT_DIR = Path("data/labins/index-clean")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# patterns
HEADER = re.compile(r"SURVEYOR\s*\(S\)", re.I)                  # typed table header, alone or merged with twp rge
YEAR = re.compile(r"^\s*1[6-9]\d\d")                            # date cells start with a year
LEAD = re.compile(r"^\s*\d{1,3}(?:\s*[,&-]\s*\d{1,3})*\s?[NSEW](?![a-z])\.?\s*")   # twp or rge token at line start
BY = re.compile(r"^.*\bby\b\.?\s*", re.I)                       # "left side by", "subdivision by"
NOT_NAME = re.compile(r"\b(index|part|parts|twp|rng|range|ranges|township|townships|diagram|page|parallel|basis|between|"
                      r"meridian|plan|island|islands|insert|not given|unknown|see|vol|lake|key|section|surveyed|base|line|lines|boundary|"
                      r"special|indices|north|south|east|west)\b", re.I)
NOT_GIVEN = re.compile(r"not given|n\.?\s?g\.?", re.I)          # placeholder in the surveyor cell
GAP = 0.3            # line gap, in median line heights, that starts a new row
PAD = 40             # slack in px around column edges

# column edges from the page itself
def edges(lines, width):
    head = [l for l in lines if HEADER.search(l["text"])]
    hb = max(l["box"][3] for l in head)
    h = head[0]
    txt, (x0, _, x1, _) = h["text"], h["box"]
    k = txt.upper().find("SURVEYOR")
    left = x0 + (x1 - x0) * k / len(txt) if k > 0 else x0 - PAD         # merged "twp rge surveyor" box is split by text position
    body = [l for l in lines if l["box"][1] > hb - 5]
    years = [l["box"][0] for l in body if YEAR.match(l["text"])]
    if years:
        right, how = float(np.percentile(years, 10)), "year cells"      # date column starts where year cells start
    else:
        d = [l for l in lines if l["text"].strip().upper().startswith("DATE") and l["box"][3] <= hb + 40]
        right, how = (min(l["box"][0] for l in d), "date header") if d else (0.43 * width, "page width")
    return hb, left, right, how

def rows_of(lines, hb):
    body = sorted([l for l in lines if l["box"][1] > hb - 5], key=lambda l: l["box"][1])
    if not body:
        return []
    med = statistics.median(l["box"][3] - l["box"][1] for l in body)
    rows, bottom = [], -1
    for l in body:
        if rows and l["box"][1] < bottom + GAP * med:
            rows[-1].append(l)                                   # second surveyor line sits just below the first
            bottom = max(bottom, l["box"][3])
        else:
            rows.append([l])
            bottom = l["box"][3]
    return rows

# text to names
def split_names(t):
    t = t.rpartition(":")[2]                                     # drop everything before a colon
    t = BY.sub("", t)
    t = re.sub(r"\([^)]*\)?|\)", " ", t)                         # parentheticals are extra info
    t = re.sub(r",\s*(jr|sr)\b", r" \1", t, flags=re.I)         # keep suffixes attached
    t = " ".join(w for w in t.split() if not re.search(r"\d", w))   # names have no digits
    parts = re.split(r"\s*(?:&|\+|;|,)\s*|(?:^|\s)and(?:\s|$)", t, flags=re.I)
    parts = [p.strip(" .,-;_") for p in parts if p and p.strip(" .,-;_")]
    out = []
    for i, p in enumerate(parts):
        last = re.findall(r"[A-Za-z]+", p)[-1] if re.search(r"[A-Za-z]", p) else ""
        if len(last) == 1:                                       # initials only, borrow the surname that follows
            nxt = next((re.findall(r"[A-Za-z]{2,}", q)[-1] for q in parts[i + 1:] if re.search(r"[A-Za-z]{3}", q)), "")
            p = f"{p}{'' if p.endswith('.') else '.'} {nxt}".strip() if nxt else p
        out.append(p)
    return out

def is_name(p):
    return bool(re.search(r"[A-Za-z]{3}", p)) and not NOT_NAME.search(p) and len(p) <= 40

def line_names(row, left, right):
    names, rejects = [], []
    for l in sorted(row, key=lambda l: (l["box"][1], l["box"][0])):
        x0, _, x1, _ = l["box"]
        if x0 >= right - PAD or x1 <= left:
            continue                                             # date, variation, page, twp, rge cells
        t = l["text"].strip()
        if re.search(r"[“”\"]", t):
            rejects.append(t)                                    # quoted place names in remarks
            continue
        if x0 < left - PAD:                                      # merged with twp rge, strip them and garbled leftovers
            while LEAD.match(t):
                t = LEAD.sub("", t, 1)
            t = re.sub(r"^(?:[A-Z]{1,2}[NSEW]\s+)+", "", t)
        t = re.sub(r"\b1[6-9]\d\d(?:-\d+)?\b", "", t)            # date merged into the line
        t = re.sub(r"\d+\s*°[^A-Za-z]*[EWew]?\b", "", t)         # variation merged into the line
        for p in split_names(t):
            (names if is_name(p) else rejects).append(p)
    return names, rejects

# extract rows
rows, rej, n, how = [], [], {"typed": 0, "no header": 0, "duplicate": 0}, {}
for f in sorted(OCR_DIR.glob("vol*.json")):
    seen = set()
    for p in json.loads(f.read_text(encoding="utf-8")):
        if p["tag"] != "index":
            continue
        key = hashlib.md5("\n".join(l["text"] for l in p["lines"]).encode()).hexdigest()
        if key in seen:
            n["duplicate"] += 1                                  # some pdfs repeat their front pages
            continue
        seen.add(key)
        if not any(HEADER.search(l["text"]) for l in p["lines"]):
            n["no header"] += 1                                  # handwritten index page and covers
            continue
        n["typed"] += 1
        hb, left, right, h = edges(p["lines"], p["width"])
        how[h] = how.get(h, 0) + 1
        for i, r in enumerate(rows_of(p["lines"], hb)):
            names, bad = line_names(r, left, right)
            names = list(dict.fromkeys(names))                   # set per row, order kept
            ng = [b for b in bad if NOT_GIVEN.fullmatch(b.strip())]   # surveyor cell reads "Not Given"
            bad = [b for b in bad if b not in ng]
            rows.append({"vol": f.stem, "page": p["page"], "row": i, "surveyors": " | ".join(names), "n_surveyors": len(names), "not_given": bool(ng),
                         "row_text": " | ".join(l["text"] for l in sorted(r, key=lambda l: (l["box"][1], l["box"][0])))})
            rej += [{"vol": f.stem, "page": p["page"], "row": i, "rejected": b} for b in bad]

# save
df, rejects = pd.DataFrame(rows), pd.DataFrame(rej)
missing = df[df["n_surveyors"] == 0]                             # not given, blanks, remark rows
df = df[df["n_surveyors"] > 0]
df.to_csv(OUT_DIR / "surveyor_rows.csv", index=False)
missing.to_csv(OUT_DIR / "surveyor_missing.csv", index=False)
rejects.to_csv(OUT_DIR / "surveyor_rejects.csv", index=False)
print(n, "| right edge from:", how)
print(len(df), "rows with a name;", (df["n_surveyors"] > 1).sum(), "with more than one;", len(missing), "in missing,", missing["not_given"].sum(), "of them not given;", len(rejects), "rejected pieces")
rejects["rejected"].value_counts().head(25)

{'typed': 366, 'no header': 92, 'duplicate': 14} | right edge from: {'year cells': 346, 'date header': 14, 'page width': 6}
3784 rows with a name; 411 with more than one; 80 in missing, 35 of them not given; 86 rejected pieces


rejected
Index                           8
Index to Twp Rng                8
Index to Twp Rng – Page         8
Special Indices                 6
Part of Twp Rng Diagram Page    6
Not Given Not Given             4
Basis Parallel West             4
Range                           4
Page                            3
Insert Page                     2
East                            2
Diagram Page                    2
Index to Twp Rng Page           2
Basis Parallel                  1
Meridian between Ranges         1
S                               1
Basis Parallel Range            1
Between Townships               1
South                           1
Basis Parallel South            1
Range Range Range Range         1
Range Range Range               1
Plan of Key Biscayne            1
J                               1
C                               1
Name: count, dtype: int64

In [10]:
# imports
import re, unicodedata
from difflib import SequenceMatcher
from pathlib import Path
import pandas as pd

# set dir
OUT_DIR = Path("data/labins/index-clean")

# config
SURNAME_MIN = 0.80    # surname ratio to join a cluster
MIN_LEN = 5           # shorter surname needs this many letters to match fuzzily
TITLES = {"col", "judge", "capt", "maj", "gen", "hon", "dr", "mr"}
SUFFIX = {"jr", "sr", "jun", "sen"}
ABBR = {"jno": "john", "jas": "james", "wm": "william", "thos": "thomas", "chas": "charles", "benj": "benjamin",
        "geo": "george", "saml": "samuel", "robt": "robert", "jos": "joseph", "danl": "daniel", "edw": "edward"}

# frequency table, one count per row a name appears in
rows = pd.read_csv(OUT_DIR / "surveyor_rows.csv", keep_default_na=False)
ment = rows.assign(surveyor=rows["surveyors"].apply(lambda s: s.split(" | "))).explode("surveyor")
ment = ment[ment["surveyor"] != ""]
freq = (ment.groupby("surveyor").agg(n=("vol", "size"), n_vols=("vol", "nunique"), first_vol=("vol", "min"))
        .reset_index().sort_values(["n", "surveyor"], ascending=[False, True]).reset_index(drop=True))
freq.to_csv(OUT_DIR / "surveyor_freq.csv", index=False)

# name parts
def parse(name):
    s = unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode().lower().replace(".", " ")
    toks = [t for t in re.findall(r"[a-z]+", s) if t not in TITLES | SUFFIX]
    return (toks[:-1], toks[-1]) if toks else ([], "")

def subseq(s, l):
    it = iter(l)
    return all(c in it for c in s)

def tok_ok(a, b):
    a, b = ABBR.get(a, a), ABBR.get(b, b)
    if a == b:
        return True
    if len(a) == 1 or len(b) == 1:
        return a[0] == b[0]                                      # initial against a full or abbreviated name
    s, l = sorted([a, b], key=len)
    return l.startswith(s) or (s[0] == l[0] and subseq(s, l))   # benj, wm, chas

def given_ok(A, B):
    return not A or not B or all(tok_ok(a, b) for a, b in zip(A, B))   # a missing given name or middle initial is fine

def sur_sim(a, b):
    if a == b:
        return 1.0
    r = SequenceMatcher(None, a, b).ratio()
    return r if min(len(a), len(b)) >= MIN_LEN else 0.0

# surname groups, more common surnames are parents
freq["given"], freq["sur"] = zip(*freq["surveyor"].map(parse))
sur_n = freq.groupby("sur")["n"].sum().sort_values(ascending=False)
sur_root = {}
for i, s in enumerate(sur_n.index):
    best = max(((sur_sim(s, t), -j, t) for j, t in enumerate(sur_n.index[:i])), default=(0, 0, None))
    sur_root[s] = sur_root[best[2]] if best[0] >= SURNAME_MIN else s    # chain to the root surname

# parent inside each group
freq["root_sur"] = freq["sur"].map(sur_root)
freq["parent"], freq["similarity"], freq["match"] = "", float("nan"), ""
for _, g in freq.groupby("root_sur"):
    idx = list(g.assign(full=g["given"].map(bool)).sort_values("full", ascending=False, kind="stable").index)   # full names lead, surname only entries trail
    for k, i in enumerate(idx[1:], 1):
        cand = []
        for j in (idx[:k] if freq.at[i, "given"] else []):
            same = (freq.at[i, "given"] == freq.at[j, "given"]) and (freq.at[i, "sur"] == freq.at[j, "sur"])
            ss = sur_sim(freq.at[i, "sur"], freq.at[j, "sur"])
            if same:
                cand.append((3, 1.0, -j, j, "same after normalizing"))
            elif ss >= SURNAME_MIN and given_ok(freq.at[i, "given"], freq.at[j, "given"]):
                cand.append((2, ss, -j, j, "given names compatible"))
        if cand:
            _, ss, _, j, m = max(cand)
        else:
            j, m = idx[0], "surname group" if freq.at[i, "given"] else "surname only"   # different given names, a person to distinguish
            ss = SequenceMatcher(None, freq.at[i, "sur"], freq.at[j, "sur"]).ratio()
        freq.at[i, "parent"], freq.at[i, "similarity"], freq.at[i, "match"] = freq.at[j, "surveyor"], round(ss, 3), m

# root, depth, cluster totals
par = dict(zip(freq["surveyor"], freq["parent"]))
def root_depth(s):
    d = 0
    while par[s]:
        s, d = par[s], d + 1
    return s, d
freq["root"], freq["depth"] = zip(*freq["surveyor"].map(root_depth))
freq["cluster_size"] = freq.groupby("root")["surveyor"].transform("size")
freq["cluster_n"] = freq.groupby("root")["n"].transform("sum")
clusters = (freq[["surveyor", "n", "n_vols", "first_vol", "parent", "similarity", "match", "root", "depth", "cluster_size", "cluster_n"]]
            .sort_values(["cluster_n", "root", "depth", "n"], ascending=[False, True, True, False]).reset_index(drop=True))
clusters.to_csv(OUT_DIR / f"surveyor_clusters_{SURNAME_MIN:.2f}.csv", index=False)   # one file per threshold
print(len(freq), "variants,", clusters["root"].nunique(), "clusters,", (clusters["cluster_size"] > 1).sum(), "variants in multi-name clusters")
clusters

224 variants, 105 clusters, 177 variants in multi-name clusters


,surveyor,n,n_vols,first_vol,parent,similarity,match,root,depth,cluster_size,cluster_n
0,B. Clements,202,17,vol013,,NaN,,B. Clements,0,9,551
1,Benj. Clements,100,10,vol001,B. Clements,1.000,given names compatible,B. Clements,1,9,551
2,J. B. Clements,89,10,vol013,B. Clements,1.000,surname group,B. Clements,1,9,551
3,H. B. Clements,48,2,vol029,B. Clements,1.000,surname group,B. Clements,1,9,551
4,Benjamin Clements,29,4,vol061,B. Clements,1.000,given names compatible,B. Clements,1,9,551
5,Clements,3,3,vol011,B. Clements,1.000,surname only,B. Clements,1,9,551
6,B. J. Clements,2,1,vol015,B. Clements,1.000,given names compatible,B. Clements,1,9,551
7,J.B. Clements,45,6,vol052,J. B. Clements,1.000,same after normalizing,B. Clements,2,9,551
8,H.B. Clements,33,2,vol035,H. B. Clements,1.000,same after normalizing,B. Clements,2,9,551
9,Henry Washington,297,14,vol003,,NaN,,Henry Washington,0,2,298


In [15]:
# imports
import json, re, statistics, unicodedata
from pathlib import Path
import pandas as pd
import pymupdf
import tkinter as tk
from PIL import Image, ImageDraw, ImageTk

# set dir
OCR_DIR = Path("data/labins/index-ocr")
PDF_DIR = Path("data/labins/index-pdfs")
OUT_DIR = Path("data/labins/index-clean")

# config
SURNAME_MIN = 0.80                      # picks the cluster file
CANVAS_W, CANVAS_H = 1000, 760          # page view in px
CONTEXT = 350                           # page px kept above and below the row
GAP = 0.3                               # same row banding as extraction
HEADER = re.compile(r"SURVEYOR\s*\(S\)", re.I)
CLUSTER_FILE = OUT_DIR / f"surveyor_clusters_{SURNAME_MIN:.2f}.csv"
DECISION_FILE = OUT_DIR / f"surveyor_decisions_{SURNAME_MIN:.2f}.csv"

# load
clusters = pd.read_csv(CLUSTER_FILE, keep_default_na=False)
rows = pd.read_csv(OUT_DIR / "surveyor_rows.csv", keep_default_na=False)
where = {}                                                       # every row a variant appears in
for r in rows.itertuples():
    for s in r.surveyors.split(" | "):
        where.setdefault(s, []).append((r.vol, r.page, r.row))
queue = clusters[clusters["parent"] != ""].reset_index(drop=True)   # roots have nothing to merge into
pos = {s: i for i, s in enumerate(queue["surveyor"])}
parent_of = dict(zip(clusters["surveyor"], clusters["parent"]))
members = {root: list(g["surveyor"]) for root, g in clusters.groupby("root", sort=False)}
root_of = dict(zip(clusters["surveyor"], clusters["root"]))

# resume earlier decisions, dropped if the parent changed
decisions, targets = {}, {}                                      # targets: merge into this name instead of the parent
if DECISION_FILE.exists():
    old = pd.read_csv(DECISION_FILE, keep_default_na=False)
    for s, p, d, t in zip(old["surveyor"], old["parent"], old["decision"], old["target"]):
        if d and parent_of.get(s) == p:
            decisions[s] = d
            if d == "merge" and t in parent_of:
                targets[s] = t

# canonical name follows merges up the tree
def canonical(s, seen=()):
    if not parent_of.get(s):
        return s
    d = decisions.get(s, "")
    if d == "merge" and s not in seen:
        return canonical(targets.get(s) or parent_of[s], seen + (s,))
    return s if d == "split" else ""                             # unsure or undecided

def save():
    out = pd.DataFrame({"surveyor": clusters["surveyor"], "parent": clusters["parent"],
                        "decision": clusters["surveyor"].map(lambda s: decisions.get(s, ""))})
    out["target"] = out["surveyor"].map(lambda s: targets.get(s, "") if decisions.get(s) == "merge" else "")   # blank means the parent
    out["canonical"] = out["surveyor"].map(canonical)
    out.to_csv(DECISION_FILE, index=False)

# page and row lookup
ocr = {}
def page_of(vol, page):
    if vol not in ocr:
        ocr[vol] = {p["page"]: p for p in json.loads((OCR_DIR / f"{vol}.json").read_text(encoding="utf-8"))}
    return ocr[vol][page]

def row_lines(p, i):
    hb = max(l["box"][3] for l in p["lines"] if HEADER.search(l["text"]))
    body = sorted([l for l in p["lines"] if l["box"][1] > hb - 5], key=lambda l: l["box"][1])
    med = statistics.median(l["box"][3] - l["box"][1] for l in body)
    out, bottom = [], -1
    for l in body:
        if out and l["box"][1] < bottom + GAP * med:
            out[-1].append(l)
            bottom = max(bottom, l["box"][3])
        else:
            out.append([l])
            bottom = l["box"][3]
    return out[i]

def ascii_lower(s):
    return unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode().lower()

# render the page with the row marked
def render(vol, page, row, name, full):
    p = page_of(vol, page)
    with pymupdf.open(PDF_DIR / f"{vol}.pdf") as doc:            # closed again so windows does not lock the pdf
        pg = doc[page - 1]
        z = p["width"] / pg.rect.width                           # ocr boxes are in the rendered image's pixels
        pix = pg.get_pixmap(matrix=pymupdf.Matrix(z, z))
    img = Image.frombytes("RGB", (pix.width, pix.height), pix.samples).convert("RGBA")
    r = row_lines(p, row)
    flat = lambda t: re.sub(r"[^a-z]", "", ascii_lower(t))
    toks = re.findall(r"[a-z]{3,}", ascii_lower(name))
    hit = ([l for l in r if flat(name) in flat(l["text"])]                   # the exact line first
           or [l for l in r if any(t in ascii_lower(l["text"]) for t in toks)] or r)   # then any line with a name part
    ov = Image.new("RGBA", img.size, (0, 0, 0, 0))
    d = ImageDraw.Draw(ov)
    for l in r:
        d.rectangle(l["box"], outline=(0, 110, 255, 255), width=3)                       # whole row
    for l in hit:
        d.rectangle(l["box"], fill=(255, 230, 0, 110), outline=(255, 0, 0, 255), width=4)   # line with the name
    img = Image.alpha_composite(img, ov).convert("RGB")
    if not full:
        y0 = max(0, min(l["box"][1] for l in r) - CONTEXT)
        y1 = min(img.height, max(l["box"][3] for l in r) + CONTEXT)
        img = img.crop((0, y0, img.width, y1))
    s = min(CANVAS_W / img.width, CANVAS_H / img.height)
    return img.resize((int(img.width * s), int(img.height * s)), Image.LANCZOS)

# window
root = tk.Tk()
root.title("surveyor merge")
state = {"i": next((k for k, s in enumerate(queue["surveyor"]) if s not in decisions), 0), "occ": 0, "peek": "", "full": tk.BooleanVar(value=False)}   # peek: member shown instead of the variant
left = tk.Frame(root, padx=8, pady=8)
left.pack(side="left", fill="y")
name_lbl = tk.Label(left, font=("Segoe UI", 16, "bold"), anchor="w", wraplength=330, justify="left")
par_lbl = tk.Label(left, font=("Segoe UI", 12), anchor="w", wraplength=330, justify="left")
info_lbl = tk.Label(left, font=("Segoe UI", 10), anchor="w", justify="left", wraplength=330)
dec_lbl = tk.Label(left, font=("Segoe UI", 11, "bold"), anchor="w")
lst = tk.Listbox(left, width=44, height=14, font=("Consolas", 10), exportselection=False)
stat_lbl = tk.Label(left, font=("Segoe UI", 10), anchor="w", justify="left")
canvas = tk.Canvas(root, width=CANVAS_W, height=CANVAS_H, bg="#222")
canvas.pack(side="right")

def show():
    r = queue.iloc[state["i"]]
    name = r["surveyor"]
    shown = state["peek"] or name                                                         # the page drawn is the variant or a peeked member
    occ = where[shown]
    state["occ"] %= len(occ)
    vol, page, row = occ[state["occ"]]
    name_lbl.config(text=name)
    par_lbl.config(text=f'under  {r["parent"]}')
    info_lbl.config(text=f'{r["match"]}, surname similarity {r["similarity"]}\n{r["n"]} mentions in {r["n_vols"]} vols\n'
                         + (f'PEEK {shown}{" (root)" if not parent_of[shown] else ""}: ' if state["peek"] else "") +
                         f'example {state["occ"] + 1} of {len(occ)}: {vol} page {page} row {row}')
    dec_lbl.config(text=f'decision: {decisions.get(name, "none yet")}' + (f' into {targets[name]}' if name in targets else ""))
    lst.delete(0, "end")                                                                  # cluster members and their decisions
    for s in members[root_of[name]]:
        n = clusters.loc[clusters["surveyor"] == s, "n"].iloc[0]
        lst.insert("end", f'{"> " if s == name else "* " if s == state["peek"] else "  "}{s} ({n})  {decisions.get(s, "") if parent_of[s] else "root"}{" > " + targets[s] if s in targets else ""}')
    cnt = pd.Series(decisions).value_counts() if decisions else pd.Series(dtype=int)
    stat_lbl.config(text=f'{state["i"] + 1} of {len(queue)}   merged {cnt.get("merge", 0)}   split {cnt.get("split", 0)}   '
                         f'unsure {cnt.get("unsure", 0)}   left {len(queue) - len(decisions)}')
    canvas.delete("all")
    try:
        photo = ImageTk.PhotoImage(render(vol, page, row, shown, state["full"].get()))
        canvas.create_image(0, 0, anchor="nw", image=photo)
        canvas.image = photo                                                              # keep a reference or tk drops it
    except Exception as e:
        canvas.create_text(20, 20, anchor="nw", fill="white", text=f"could not draw {vol} page {page}: {e}")

def decide(kind, target=""):
    name = queue.at[state["i"], "surveyor"]
    decisions[name] = kind
    if target:
        targets[name] = target
    else:
        targets.pop(name, None)
    save()
    state["i"], state["occ"], state["peek"] = min(state["i"] + 1, len(queue) - 1), 0, ""   # move on
    show()

def move(step):
    state["i"], state["occ"], state["peek"] = max(0, min(state["i"] + step, len(queue) - 1)), 0, ""
    show()

def other():
    state["occ"] += 1
    show()

def merge_into():
    name = queue.at[state["i"], "surveyor"]
    cands = [s for s in members[root_of[name]] if s != name and (not parent_of[s] or decisions.get(s) == "split")]   # roots and splits
    win = tk.Toplevel(root)
    win.title(f"merge {name} into")
    box = tk.Listbox(win, width=46, height=min(len(cands), 15) + 1, font=("Consolas", 10), exportselection=False)
    for s in cands:
        box.insert("end", f'{s} ({clusters.loc[clusters["surveyor"] == s, "n"].iloc[0]})  {"root" if not parent_of[s] else "split"}')
    def pick(_=None):
        sel = box.curselection()
        if sel:
            win.destroy()
            decide("merge", cands[sel[0]])
    box.bind("<Double-Button-1>", pick)
    box.bind("<Return>", pick)
    box.pack(padx=8, pady=8)
    tk.Button(win, text="merge into selected", command=pick).pack(pady=4)
    box.focus_set()

def listed(k):
    return members[root_of[queue.at[state["i"], "surveyor"]]][k]

def peek(_):                                                     # one click shows that member's page
    sel = lst.curselection()
    if sel:
        name = listed(sel[0])
        state["peek"] = "" if name == queue.at[state["i"], "surveyor"] else name
        state["occ"] = 0
        show()

def jump(e):                                                     # double click moves to that variant
    name = listed(lst.nearest(e.y))
    if name in pos:
        state["i"], state["occ"], state["peek"] = pos[name], 0, ""
        show()

def peek_root():
    state["peek"] = "" if state["peek"] else root_of[queue.at[state["i"], "surveyor"]]
    state["occ"] = 0
    show()

lst.bind("<<ListboxSelect>>", peek)
lst.bind("<Double-Button-1>", jump)
btns = tk.Frame(left)
for text, cmd, key in [("merge (m)", lambda: decide("merge"), "m"), ("split (s)", lambda: decide("split"), "s"), ("unsure (u)", lambda: decide("unsure"), "u")]:
    tk.Button(btns, text=text, width=11, command=cmd).pack(side="left", padx=2)
    root.bind(f"<{key}>", lambda e, c=cmd: c())
nav = tk.Frame(left)
tk.Button(nav, text="< back", width=11, command=lambda: move(-1)).pack(side="left", padx=2)
tk.Button(nav, text="next >", width=11, command=lambda: move(1)).pack(side="left", padx=2)
tk.Button(nav, text="other example (o)", command=other).pack(side="left", padx=2)
root.bind("<Left>", lambda e: move(-1))
root.bind("<Right>", lambda e: move(1))
root.bind("<o>", lambda e: other())
root.bind("<t>", lambda e: merge_into())
root.bind("<r>", lambda e: peek_root())
root.bind("<f>", lambda e: (state["full"].set(not state["full"].get()), show()))
for w in (name_lbl, par_lbl, info_lbl, dec_lbl, lst, btns, nav, stat_lbl):
    w.pack(fill="x", pady=4)
tk.Button(left, text="merge into... (t)", command=merge_into).pack(anchor="w", pady=2)
tk.Button(left, text="show root (r)", command=peek_root).pack(anchor="w", pady=2)
tk.Checkbutton(left, text="full page (f)", variable=state["full"], command=show).pack(anchor="w")
root.lift()
root.attributes("-topmost", True)
root.after(400, lambda: root.attributes("-topmost", False))
show()
root.mainloop()